# Sistema Multi-Agente Strands com Ferramenta de Memória AgentCore (Memória de Curto Prazo) - Usando MemoryManager

## Introdução

Este notebook demonstra como implementar um **sistema multi-agente com memória compartilhada** usando AWS AgentCore Memory e o framework Strands. Enquanto nossos exemplos anteriores focaram em memória de agente único, este notebook explora como múltiplos agentes especializados podem trabalhar juntos acessando um armazenamento de memória comum.

**NOTA: Esta é a versão do exemplo de Memória de Curto Prazo usando o MemoryManager & MemorySessionManager no lugar do MemoryClient original.**

## Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Curto Prazo Conversacional                                                       |
| Caso de uso do agente | Assistente de Planejamento de Viagens                                         |
| Framework Agêntico  | Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | Memória de Curto Prazo AgentCore, Strands Agents, Recuperação de memória via Ferramenta |
| Complexidade do exemplo | Iniciante                                                                    |


O que você aprenderá:

- Como configurar um recurso de memória compartilhada que múltiplos agentes podem acessar
- Criar agentes especializados como ferramentas com seu próprio acesso à memória
- Implementar um agente coordenador que delega para agentes especializados
- Manter contexto de conversa entre múltiplas interações de agentes

### Contexto do Cenário

Neste exemplo, vamos criar um **Sistema de Planejamento de Viagens** com:
1. Um Assistente de Reserva de Voos especializado em viagens aéreas
2. Um Assistente de Reserva de Hotéis focado em acomodações
3. Um Coordenador de Viagens que delega para esses agentes especializados

Esta abordagem demonstra como domínios complexos podem ser decompostos em agentes especializados que compartilham o mesmo armazenamento de memória.

## Arquitetura
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Pré-requisitos
- Python 3.10+
- Conta AWS com permissões apropriadas
- Role IAM AWS com permissões apropriadas para AgentCore Memory
- Acesso aos modelos do Amazon Bedrock

Vamos começar configurando nosso ambiente e criando nosso recurso de memória compartilhada!

## Passo 1: Configuração do Ambiente
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para fazer este notebook funcionar.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
import os
from datetime import datetime
from botocore.exceptions import ClientError
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

# Import memory management modules
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

Defina a região e a role com as permissões apropriadas para modelos Amazon Bedrock e AgentCore

In [ ]:
REGION = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## Passo 2: Criando Memória Compartilhada
Nesta seção, vamos criar um recurso de memória que será compartilhado entre nossos agentes especializados.

In [ ]:
memory_manager = MemoryManager(region_name=REGION)

try:
    print("Creating Memory...")
    memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")

    # Create the memory resource
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=[],  # No strategies for short-term memory
        description="Short-term memory for travel agent",
        event_expiry_days=7,  # Retention period for short-term memory
        memory_execution_role_arn=None,  # Optional for short-term memory
    )

    # Extract and print the memory ID
    memory_id = memory.id
    logger.info(f"✅ Successfully created/retrieved memory with MemoryManager:")
    logger.info(f"   Memory ID: {memory_id}")
    logger.info(f"   Memory Name: {memory.name}")
    logger.info(f"   Memory Status: {memory.status}")
except Exception as e:
    # Handle any errors during memory creation with enhanced error reporting
    logger.error(f"❌ Memory creation failed: {e}")
    logger.error(f"Error type: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    
    # Cleanup on error - delete the memory if it was partially created
    if 'memory_id' in locals():
        try:
            logger.info(f"Attempting cleanup of partially created memory: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"✅ Successfully cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"❌ Failed to clean up memory: {cleanup_error}")
    
    # Re-raise the original exception
    raise

### Entendendo Memória Compartilhada para Sistemas Multi-Agente

O recurso de memória que criamos servirá como uma base de conhecimento compartilhada para nosso sistema de planejamento de viagens. Todos os agentes lerão e escreverão neste armazenamento comum de memória, permitindo:

1. **Consistência de Conhecimento**: Todos os agentes trabalham com as mesmas informações
2. **Preservação de Contexto**: O histórico de conversa é mantido entre transições de agentes
3. **Acesso Especializado**: Cada agente terá seu próprio actor_id mas compartilhará o session_id

Esta abordagem permite que agentes especializados foquem em seus domínios enquanto se beneficiam do contexto completo da conversa.

## Passo 3: Inicializar Gerenciador de Sessão

Esta seção apresenta o MemorySessionManager para operações de memória baseadas em sessão e cria uma MemorySession para gerenciar ator e sessão

In [ ]:
# Initialize the session memory manager
session_manager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

logger.info(f"✅ Session manager initialized for memory: {memory.id}")
logger.info(f"Session manager type: {type(session_manager)}")

## Passo 4: Criar Provedor de Hook de Memória

Este passo define nossa classe customizada `MemoryHookProvider` que automatiza operações de memória. Hooks são funções especiais que executam em pontos específicos do ciclo de vida de execução de um agente. O hook de memória que estamos criando serve duas funções principais:

1. **Recuperar Memórias**: Busca automaticamente conversas passadas relevantes quando um usuário envia uma mensagem
2. **Salvar Memórias**: Armazena novas conversas após o agente responder

**MUDANÇAS PRINCIPAIS em relação à versão MemoryClient:**
- Usa MemorySession em vez de MemoryClient
- Usa objetos ConversationalMessage em vez de tuplas
- Usa add_turns() em vez de create_event()
- Usa enum MessageRole para segurança de tipos

In [ ]:
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_session: MemorySession, memory_id: str):
        self.memory_session = memory_session
        self.memory_id = memory_id
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts"""
        try:
            # Use the pre-configured memory session (no need for actor_id/session_id)
            recent_turns = self.memory_session.get_last_k_turns(k=5)
            
            if recent_turns:
                # Format conversation history for context
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        # Handle both EventMessage objects and dict formats
                        if hasattr(message, 'role') and hasattr(message, 'content'):
                            role = message['role']
                            content = message['content']
                        else:
                            role = message.get('role', 'unknown')
                            content = message.get('content', {}).get('text', '')
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                logger.info(f"Context from memory: {context}")
                
                # Add context to agent's system prompt
                event.agent.system_prompt += f"\n\nRecent conversation history:\n{context}\n\nContinue the conversation naturally based on this context."
                logger.info(f"✅ Loaded {len(recent_turns)} recent conversation turns")
            else:
                logger.info("No previous conversation history found")
                
        except Exception as e:
            logger.error(f"Failed to load conversation history: {e}")
    
    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory using MemorySession"""
        messages = event.agent.messages
        try:
            if messages and len(messages) > 0 and messages[-1]["content"][0].get("text"):
                message_text = messages[-1]["content"][0]["text"]
                message_role = MessageRole.USER if messages[-1]["role"] == "user" else MessageRole.ASSISTANT
                
                # Use memory session instance (no need to pass actor_id/session_id)
                result = self.memory_session.add_turns(
                    messages=[ConversationalMessage(message_text, message_role)]
                )
                
                event_id = result['eventId']
                logger.info(f"✅ Stored message with Event ID: {event_id}, Role: {message_role.value}")
                
        except Exception as e:
            logger.error(f"Memory save error: {e}")
            import traceback
            logger.error(f"Full traceback: {traceback.format_exc()}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        # Register memory hooks
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## Passo 5: Criar Arquitetura Multi-Agente com Strands Agents
Nesta seção, vamos criar nosso sistema multi-agente com agentes especializados para reservas de voos e hotéis, ambos compartilhando acesso ao nosso recurso de memória.

In [ ]:
# Import the necessary components
from strands import Agent, tool

In [ ]:
# Create unique actor IDs for each specialized agent but share the session ID
FLIGHT_ACTOR_ID = f"flight-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
HOTEL_ACTOR_ID = f"hotel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
SESSION_ID = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

### Criando Agentes Especializados com Acesso à Memória

A seguir, vamos definir prompts de sistema para nossos agentes especializados. Cada prompt inclui os parâmetros de memória em um formato que o agente pode interpretar:

In [ ]:
# System prompt for the hotel booking specialist
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# System prompt for the flight booking specialist
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

### Implementando Ferramentas de Agente
Agora vamos implementar nossos agentes especializados como ferramentas que podem ser usadas pelo agente coordenador:

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        # Create a memory session for the booking assistant
        memory_session = session_manager.create_memory_session(
            actor_id=FLIGHT_ACTOR_ID, 
            session_id=SESSION_ID
        )
        flight_memory_hooks = ShortTermMemoryHook(memory_session, memory_id)
        
        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": FLIGHT_ACTOR_ID, "session_id": SESSION_ID}
        )

        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"

@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        # Create a memory session for the booking assistant
        memory_session = session_manager.create_memory_session(
            actor_id=HOTEL_ACTOR_ID, 
            session_id=SESSION_ID
        )

        hotel_memory_hooks = ShortTermMemoryHook(memory_session, memory_id)

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": HOTEL_ACTOR_ID, "session_id": SESSION_ID}
        )
        
        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### Criando o Agente Coordenador

Finalmente, vamos criar o agente principal de planejamento de viagens que coordena entre essas ferramentas especializadas:

In [ ]:
# System prompt for the coordinator agent
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

#### Seu Sistema Multi-Agente está pronto!!

## Vamos testar o Agente.

Vamos testar nosso sistema multi-agente com um cenário de planejamento de viagens:

In [ ]:
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
response = travel_agent("I would only like to focus on the flight at the moment. direct flimid-range, city center, pool, standard room")

## Testando Persistência de Memória

Para testar se nosso sistema de memória está funcionando corretamente, vamos criar uma nova instância do agente de viagens e ver se ele consegue acessar as informações previamente armazenadas:

In [ ]:
# Create a new instance of the travel agent
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

# Ask about previous conversations
new_travel_agent("Can you remind me about flights talked about before?")

## Resumo

Neste notebook, demonstramos:

1. Como criar um recurso de memória compartilhada para múltiplos agentes
2. Como implementar agentes especializados como ferramentas com acesso à memória
3. Como coordenar entre múltiplos agentes mantendo contexto de conversa
4. Como a memória persiste entre diferentes instâncias de agentes

Esta arquitetura multi-agente com memória compartilhada fornece uma abordagem poderosa para construir sistemas de IA conversacional complexos que podem lidar com domínios especializados mantendo uma experiência de usuário coesa.

## Limpeza
Vamos deletar a memória para limpar os recursos usados neste notebook.

In [ ]:
# Uncomment to delete memory resource using MemoryManager
# try:
#     memory_manager.delete_memory(memory_id)
#     logger.info(f"✅ Deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Failed to delete memory: {e}")